In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Board state encoding ──────────────────────────────────────────────────────

def clues_to_board_state(clues: torch.Tensor, hidden: torch.Tensor) -> torch.Tensor:
    """
    Convert raw clue values and hidden mask into the 10-channel board state
    the energy model expects.

    Args:
        clues:  (B, 1, H, W)  values in {-1 (hidden), 0-8}
        hidden: (B, 1, H, W)  1 = cell is hidden, 0 = revealed

    Returns:
        board_state (B, 10, H, W):
            channels 0-8  one-hot of the revealed clue value (0 on hidden cells)
            channel  9    hidden mask
    """
    B, _, H, W = clues.shape

    # Clamp -1 (hidden) to 0 before one-hot so indexing is safe;
    # we zero out hidden cells afterwards anyway.
    clue_vals = clues.squeeze(1).long().clamp(min=0)       # (B, H, W)

    onehot = F.one_hot(clue_vals, num_classes=9)           # (B, H, W, 9)
    onehot = onehot.permute(0, 3, 1, 2).float()            # (B, 9, H, W)

    # Hidden cells carry no clue information — zero them out
    reveal_mask = 1.0 - hidden                             # (B, 1, H, W)
    onehot = onehot * reveal_mask

    return torch.cat([onehot, hidden], dim=1)              # (B, 10, H, W)

# ── Replay Buffer ─────────────────────────────────────────────────────────────
class ReplayBuffer:
    """
    Stores previously-generated candidate logits so the model must maintain
    a well-shaped energy valley near the solution, not just near the noise init.
    Follows Du & Mordatch 2019 as cited in the paper (ref [51]).
    """
    def __init__(self, max_size: int = 10_000, replay_prob: float = 0.95):
        self.buffer      = []
        self.max_size    = max_size
        self.replay_prob = replay_prob  # probability of drawing from buffer vs fresh noise

    def sample(self, B: int, H: int, W: int, device: torch.device) -> torch.Tensor:
        if len(self.buffer) < B or random.random() > self.replay_prob:
            # Fall back to fresh noise (also how the buffer gets seeded initially)
            return torch.randn(B, 1, H, W, device=device) * 0.1

        # Draw B entries from the buffer
        indices  = random.sample(range(len(self.buffer)), B)
        batch    = torch.stack([self.buffer[i] for i in indices]).to(device)
        return batch

    def push(self, logits: torch.Tensor):
        """Store the final candidate logits from this training step."""
        for i in range(logits.shape[0]):
            entry = logits[i].detach().cpu()
            if len(self.buffer) >= self.max_size:
                # Randomly replace an existing entry (reservoir-style)
                self.buffer[random.randint(0, self.max_size - 1)] = entry
            else:
                self.buffer.append(entry)

# ── Energy model ──────────────────────────────────────────────────────────────

class MinesweeperEnergyModel(nn.Module):
    """
    Assigns a scalar energy to every (board_state, candidate_mines) pair.
    Lower energy = higher compatibility = this mine pattern fits the clues better.

    Input channels (11 total):
        0-8   one-hot clue values (revealed cells only)
        9     hidden mask
        10    candidate mine probabilities in [0, 1]

    GroupNorm after each conv keeps activations at O(1) scale regardless of
    board size, which is essential for stable gradient magnitudes during the
    inner optimisation loop.
    """

    def __init__(self, board_width: int = 30, board_height: int = 16,
                 hidden_dim: int = 128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(11,         hidden_dim, kernel_size=5, padding=2),
            nn.GroupNorm(8, hidden_dim),
            nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GroupNorm(8, hidden_dim),
            nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GroupNorm(8, hidden_dim),
            nn.SiLU(),
            nn.Conv2d(hidden_dim, 1,          kernel_size=5, padding=2),
        )

    def forward(self, board_state: torch.Tensor,
                candidate_mines: torch.Tensor) -> torch.Tensor:
        """
        Args:
            board_state:     (B, 10, H, W)
            candidate_mines: (B,  1, H, W)  mine probabilities in [0, 1]

        Returns:
            energy: (B,)  scalar energy per board
        """
        x = torch.cat([board_state, candidate_mines], dim=1)  # (B, 11, H, W)
        return self.net(x).mean(dim=[1, 2, 3])                 # (B,)


# ── Energy sampler (inference) ────────────────────────────────────────────────

class EnergySampler:
    """
    Refines a random candidate mine map by gradient-descending the energy
    landscape. At inference, we use pure gradient descent (noise=0) to
    settle into the absolute minimum of the energy valley.
    """

    def __init__(self, model: nn.Module, num_steps: int = 15,
                 alpha: float = 1.0, noise_std: float = 0.0):
        self.model     = model
        self.num_steps = num_steps
        self.alpha     = alpha
        self.noise_std = noise_std

    def sample(self, board_state: torch.Tensor,
               inference_steps: int | None = None) -> torch.Tensor:

        # FORCE PyTorch to track gradients, even if the caller used torch.no_grad()
        with torch.enable_grad():
            self.model.eval()

            # Freeze model weights — we only optimise the candidate logits
            for param in self.model.parameters():
                param.requires_grad_(False)

            B, _, H, W = board_state.shape
            device = board_state.device
            steps  = inference_steps if inference_steps is not None else self.num_steps

            # 1. Scale down step size for higher compute to avoid overshooting
            # Assuming the model was trained with max_steps=5
            train_max_steps = 5.0
            step_size = self.alpha * (train_max_steps / max(train_max_steps, float(steps)))

            candidate_logits = torch.randn(B, 1, H, W, device=device) * 0.1
            candidate_logits.requires_grad_(True)

            for _ in range(steps):
                candidate_probs = torch.sigmoid(candidate_logits)
                energy = self.model(board_state, candidate_probs)

                # Get gradient exactly like in training
                grad = torch.autograd.grad(energy.sum(), candidate_logits)[0]

                with torch.no_grad():
                    # Manual update
                    if self.noise_std > 0:
                        noise = self.noise_std * torch.randn_like(candidate_logits)
                    else:
                        noise = 0.0

                    candidate_logits = candidate_logits - step_size * grad + noise

                    # 2. Clamp logits to prevent sigmoid vanishing gradients!
                    candidate_logits = torch.clamp(candidate_logits, min=-10.0, max=10.0)

                # Re-enable gradient tracking for the next inner step
                candidate_logits.requires_grad_(True)

            # Re-enable model gradients for the next training step if needed
            for param in self.model.parameters():
                param.requires_grad_(True)

            return torch.sigmoid(candidate_logits).detach()

In [ ]:
import torch
import torch.nn.functional as F
import random
import os
import matplotlib.pyplot as plt
import numpy as np


# ── Board generation ──────────────────────────────────────────────────────────

def _place_mines(height: int, width: int, num_mines: int,
                 safe_r: int, safe_c: int):
    """
    Place mines randomly, excluding (safe_r, safe_c) and its 8 neighbours.
    Returns (mines, numbers) tensors of shape (H, W).
    """
    kernel = torch.ones(1, 1, 3, 3)
    kernel[0, 0, 1, 1] = 0

    forbidden = set()
    for dr in range(-1, 2):
        for dc in range(-1, 2):
            nr, nc = safe_r + dr, safe_c + dc
            if 0 <= nr < height and 0 <= nc < width:
                forbidden.add(nr * width + nc)

    available = [i for i in range(height * width) if i not in forbidden]
    chosen    = torch.tensor(random.sample(available, num_mines))

    flat       = torch.zeros(height * width)
    flat[chosen] = 1
    mines   = flat.reshape(height, width)
    numbers = F.conv2d(
        mines.view(1, 1, height, width), kernel, padding=1
    ).squeeze()

    return mines, numbers


def generate_board(height: int = 16, width: int = 30,
                   num_mines: int = 99, max_clicks: int = 5):
    """
    Generate one Minesweeper board in a partially-revealed state.

    The first click is chosen randomly from ANY cell, then mines are placed
    around it (guaranteeing it is safe). Subsequent clicks simulate a player
    revealing safe frontier cells.

    Returns:
        clues:  (1, H, W)  float — revealed cell values (0-8), -1 for hidden
        mines:  (1, H, W)  float — ground-truth mine locations (0 or 1)
        hidden: (1, H, W)  float — 1 = hidden, 0 = revealed
    """
    kernel = torch.ones(1, 1, 3, 3)
    kernel[0, 0, 1, 1] = 0

    # First click: any cell — mines placed around it afterwards
    first_r = random.randint(0, height - 1)
    first_c = random.randint(0, width  - 1)
    mines, numbers = _place_mines(height, width, num_mines, first_r, first_c)

    hidden = torch.ones(height, width, dtype=torch.bool)

    def neighbors(r, c):
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if (dr or dc) and 0 <= r + dr < height and 0 <= c + dc < width:
                    yield r + dr, c + dc

    def click(r, c):
        stack = [(r, c)]
        while stack:
            r, c = stack.pop()
            if not hidden[r, c] or mines[r, c]:
                continue
            hidden[r, c] = False
            if numbers[r, c] == 0:
                stack += [(nr, nc) for nr, nc in neighbors(r, c)
                          if hidden[nr, nc]]

    click(first_r, first_c)

    # Simulate additional safe clicks along the frontier
    for _ in range(max_clicks - 1):
        adj_revealed = F.conv2d(
            (~hidden).float().view(1, 1, height, width), kernel, padding=1
        ).squeeze() > 0
        frontier = (hidden & adj_revealed).nonzero().tolist()

        safe     = [(r, c) for r, c in frontier if not mines[r, c]]
        fallback = (hidden & (mines == 0)).nonzero().tolist()

        if safe:
            click(*random.choice(safe))
        elif fallback:
            click(*random.choice(fallback))
        else:
            break

    clues = torch.where(hidden, torch.tensor(-1.0), numbers)
    return clues.unsqueeze(0), mines.unsqueeze(0), hidden.float().unsqueeze(0)


# ── Visualisation ─────────────────────────────────────────────────────────────

def save_board_visual(clues: torch.Tensor, mines: torch.Tensor,
                      hidden: torch.Tensor, filepath: str):
    """Save a PNG visualisation of one board for debugging / sanity-checking."""
    clues  = clues.squeeze().numpy()
    mines  = mines.squeeze().numpy()
    hidden = hidden.squeeze().numpy()

    H, W = clues.shape
    fig, ax = plt.subplots(figsize=(W * 0.6, H * 0.6))

    ax.set_xticks(np.arange(W + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(H + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_xticks([]); ax.set_yticks([])

    colors = ["blue", "green", "red", "darkblue", "maroon", "cyan", "black", "gray"]

    for r in range(H):
        for c in range(W):
            if hidden[r, c]:
                face = "salmon" if mines[r, c] else "darkgray"
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           facecolor=face))
                if mines[r, c]:
                    ax.text(c, r, "M", va="center", ha="center",
                            color="darkred", fontweight="bold", fontsize=11)
            else:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           facecolor="lightgray"))
                val = int(clues[r, c])
                if val > 0:
                    ax.text(c, r, str(val), va="center", ha="center",
                            color=colors[val - 1], fontweight="bold", fontsize=11)

    ax.set_xlim(-0.5, W - 0.5)
    ax.set_ylim(-0.5, H - 0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.close(fig)


# ── Dataset creation ──────────────────────────────────────────────────────────

def create_and_save_dataset(filepath: str = "minesweeper_train.pt",
                            num_samples: int = 100_000,
                            num_visuals:  int = 5,
                            height:       int = 16,
                            width:        int = 30,
                            num_mines:    int = 99,
                            max_clicks:   int = 5):
    """
    Generate `num_samples` boards and save as a list of (clues, mines, hidden)
    tuples in a .pt file.  Also saves `num_visuals` PNG samples for inspection.
    """
    os.makedirs("visuals", exist_ok=True)
    dataset = []

    print(f"Generating {num_samples:,} boards "
          f"({height}x{width}, {num_mines} mines)...")

    for i in range(num_samples):
        board_data = generate_board(height, width, num_mines, max_clicks)
        dataset.append(board_data)

        if i < num_visuals:
            clues, mines, hidden = board_data
            save_board_visual(clues, mines, hidden,
                              os.path.join("visuals", f"sample_{i+1}.png"))

        if (i + 1) % 5_000 == 0:
            print(f"  {i+1:>8,} / {num_samples:,}")

    torch.save(dataset, filepath)
    print(f"\nSaved {num_samples:,} boards → {filepath}")
    print(f"Saved {num_visuals} sample visuals → visuals/")

In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from model import MinesweeperEnergyModel, clues_to_board_state
# ── Dataset ───────────────────────────────────────────────────────────────────
class MinesweeperDataset(Dataset):
    """Wraps the list-of-tuples .pt file produced by generate_dataset.py."""
    def __init__(self, filepath: str):
        raw = torch.load(filepath, weights_only=True)
        self.clues  = torch.stack([r[0] for r in raw])   # (N, 1, H, W)
        self.mines  = torch.stack([r[1] for r in raw])   # (N, 1, H, W)
        self.hidden = torch.stack([r[2] for r in raw])   # (N, 1, H, W)
    def __len__(self):
        return len(self.clues)
    def __getitem__(self, idx):
        return self.clues[idx], self.mines[idx], self.hidden[idx]
# ── Alpha calibration ─────────────────────────────────────────────────────────
def calibrate_alpha(model: nn.Module, dataset: MinesweeperDataset,
                    device: torch.device, n_boards: int = 100):
    """
    Measure actual gradient magnitudes on real boards to set alpha so that
    each inner step moves candidate logits by ~1 unit (enough to cross a
    sigmoid decision boundary without overshooting).
    A good alpha satisfies:   alpha * grad_magnitude ≈ 1.0
    We use the median gradient magnitude as the anchor and derive a symmetric
    log-factor from the 10th/90th percentile spread.
    Returns:
        alpha_base   — median-calibrated step size
        alpha_factor — multiplicative randomisation range (paper uses 2.0 for NLP;
                       yours will be derived from your actual gradient distribution)
    """
    model.eval()
    loader     = DataLoader(dataset, batch_size=1, shuffle=True)
    grad_norms = []
    print(f"Calibrating alpha from {n_boards} boards...")
    for i, (clues, mines, hidden) in enumerate(loader):
        if i >= n_boards:
            break
        clues, mines, hidden = (t.to(device) for t in (clues, mines, hidden))
        board_state = clues_to_board_state(clues, hidden)
        candidate_logits = torch.randn(1, 1, 16, 30, device=device) * 0.1
        candidate_logits.requires_grad_(True)
        probs  = torch.sigmoid(candidate_logits)
        energy = model(board_state, probs)
        grad   = torch.autograd.grad(energy.sum(), candidate_logits)[0]
        grad_norms.append(grad.abs().mean().item())
    grad_norms = torch.tensor(grad_norms)
    p10 = grad_norms.quantile(0.10).item()
    p50 = grad_norms.quantile(0.50).item()
    p90 = grad_norms.quantile(0.90).item()
    alpha_base   = 1.0 / p50 if p50 > 1e-8 else 1.0
    alpha_factor = ((1.0 / p10) / alpha_base) ** 0.5 if p10 > 1e-8 else 2.0
    alpha_factor = max(1.1, min(alpha_factor, 10.0))   # clamp to sensible range
    print(f"  Gradient magnitudes — p10:{p10:.5f}  p50:{p50:.5f}  p90:{p90:.5f}")
    print(f"  alpha_base={alpha_base:.4f}  alpha_factor={alpha_factor:.4f}")
    print(f"  Effective range: [{alpha_base/alpha_factor:.4f}, "
          f"{alpha_base*alpha_factor:.4f}]")
    model.train()
    return alpha_base, alpha_factor
# ── Training step ─────────────────────────────────────────────────────────────
def ebm_train_step(
    model, clues, mines, hidden,
    *,
    replay_buffer: ReplayBuffer,
    alpha_base, alpha_factor,
    noise_std=0.05, mine_weight=7.0,
    min_steps=1, max_steps=5,
) -> torch.Tensor:
    """
    One EBM training step:
    1. Initialise candidate logits from N(0, 0.1)
    2. Run a random number of gradient-descent steps on the energy,
       each with a randomly sampled alpha (log-uniform in
       [alpha_base/alpha_factor, alpha_base*alpha_factor])
    3. Compute weighted BCE loss against ground-truth mines at the final step
    4. Return scalar loss — caller does .backward() + optimiser.step()
    create_graph=True in autograd.grad keeps the full unrolled computation
    graph alive so the outer loss can teach the energy model what landscape
    would have guided the prediction to the right answer.
    """
    B, _, H, W = clues.shape
    device = clues.device
    board_state = clues_to_board_state(clues, hidden)

    # Initialize from replay buffer instead of always fresh noise
    candidate_logits = replay_buffer.sample(B, H, W, device)
    candidate_logits.requires_grad_(True)

    steps = random.randint(min_steps, max_steps)
    for _ in range(steps):
        candidate_probs = torch.sigmoid(candidate_logits)
        energy = model(board_state, candidate_probs)
        grad = torch.autograd.grad(
            energy.sum(), candidate_logits, create_graph=True,
        )[0]
        log_factor       = torch.empty(1).uniform_(-1.0, 1.0).item()
        alpha            = alpha_base * (alpha_factor ** log_factor)
        noise            = noise_std * torch.randn_like(candidate_logits)
        candidate_logits = candidate_logits - alpha * grad + noise

    # Push final candidates back into the buffer for future steps
    replay_buffer.push(candidate_logits)

    final_probs = torch.sigmoid(candidate_logits)
    pixel_loss  = F.binary_cross_entropy(final_probs, mines, reduction="none")
    weights     = torch.where(
        mines.bool(),
        torch.tensor(mine_weight, device=device),
        torch.tensor(1.0,         device=device),
    )
    return (pixel_loss * weights * hidden).sum() / hidden.sum().clamp(min=1)
# ── Sanity check ──────────────────────────────────────────────────────────────
def sanity_check(model: nn.Module, device: torch.device):
    """
    Quick mid-training diagnostic: E(true mines) must be < E(all mines).
    If this is inverted the energy landscape is still wrong.
    """
    model.eval()
    clues, mines, hidden = generate_board()
    clues  = clues.unsqueeze(0).to(device)
    mines  = mines.unsqueeze(0).to(device)
    hidden = hidden.unsqueeze(0).to(device)
    board  = clues_to_board_state(clues, hidden)
    with torch.no_grad():
        # Only place mines on hidden cells — revealed cells are known safe
        all_mines = hidden.clone()
        e_true    = model(board, mines).item()
        e_all     = model(board, all_mines).item()
    status = "GOOD" if e_true < e_all else "INVERTED — still learning"
    print(f"  [Sanity] E(true)={e_true:.4f}  E(all_mines)={e_all:.4f}  [{status}]")
    model.train()
# ── Main training loop ────────────────────────────────────────────────────────
def train(
    data_path:      str   = "minesweeper_train.pt",
    checkpoint_dir: str   = "checkpoints",
    *,
    epochs:         int   = 50,
    batch_size:     int   = 128,
    lr:             float = 3e-4,
    noise_std:      float = 0.05,
    mine_weight:    float = 7.0,
    min_steps:      int   = 1,
    max_steps:      int   = 5,
    log_every:      int   = 50,
    save_every:     int   = 5,
    device_str:     str   = "auto",
):
    device = (torch.device("cuda" if torch.cuda.is_available() else "cpu")
              if device_str == "auto" else torch.device(device_str))
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Training on {device}\n")
    dataset    = MinesweeperDataset(data_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=4, pin_memory=True)
    model     = MinesweeperEnergyModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(dataloader), eta_min=lr * 0.05
    )
    # Calibrate alpha before training starts
    alpha_base, alpha_factor = 1.0, 2.0
    replay_buffer = ReplayBuffer(max_size=10_000, replay_prob=0.95)
    print()
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for step, (clues, mines, hidden) in enumerate(dataloader, 1):
            clues, mines, hidden = (t.to(device) for t in (clues, mines, hidden))
            optimizer.zero_grad()
            loss = ebm_train_step(
                model, clues, mines, hidden,
                replay_buffer=replay_buffer,
                alpha_base   = alpha_base,
                alpha_factor = alpha_factor,
                noise_std    = noise_std,
                mine_weight  = mine_weight,
                min_steps    = min_steps,
                max_steps    = max_steps,
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            if step % log_every == 0:
                avg = epoch_loss / step
                print(f"Epoch {epoch:>3} | {step:>5}/{len(dataloader)} "
                      f"| loss {loss.item():.4f} | avg {avg:.4f} "
                      f"| lr {scheduler.get_last_lr()[0]:.2e}")
        avg_epoch = epoch_loss / len(dataloader)
        print(f"\n── Epoch {epoch} complete — avg loss: {avg_epoch:.4f}")
        sanity_check(model, device)
        print()
        if epoch % save_every == 0:
            path = os.path.join(checkpoint_dir, f"ebm_epoch{epoch:03d}.pt")
            torch.save({
                "epoch":        epoch,
                "model":        model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "scheduler":    scheduler.state_dict(),
                "avg_loss":     avg_epoch,
                "alpha_base":   alpha_base,
                "alpha_factor": alpha_factor,
                "replay_buffer":  replay_buffer.buffer,
            }, path)
            print(f"Saved checkpoint → {path}\n")
    return model



In [ ]:
import random
from dataclasses import dataclass, field
from typing import Optional

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# from model import MinesweeperEnergyModel, EnergySampler, clues_to_board_state


# ── Game engine ───────────────────────────────────────────────────────────────

class MinesweeperGame:
    """
    Interactive Minesweeper game engine.

    Mines are placed on the FIRST click, guaranteeing the clicked cell and
    all its neighbours are always mine-free — matching real Minesweeper behaviour.
    """

    def __init__(self, height: int = 16, width: int = 30, num_mines: int = 99):
        self.height    = height
        self.width     = width
        self.num_mines = num_mines
        self.reset()

    def reset(self):
        self.mines          = torch.zeros(self.height, self.width)
        self.numbers        = torch.zeros(self.height, self.width)
        self.hidden         = torch.ones(self.height, self.width, dtype=torch.bool)
        self.exploded       = False
        self.move_count     = 0
        self._mines_placed  = False

    # ── mine placement ────────────────────────────────────────────────────────

    def _place_mines(self, safe_r: int, safe_c: int):
        """Place mines after the first click, keeping (safe_r, safe_c) safe."""
        kernel = torch.ones(1, 1, 3, 3)
        kernel[0, 0, 1, 1] = 0

        forbidden = set()
        for dr in range(-1, 2):
            for dc in range(-1, 2):
                nr, nc = safe_r + dr, safe_c + dc
                if 0 <= nr < self.height and 0 <= nc < self.width:
                    forbidden.add(nr * self.width + nc)

        available = [i for i in range(self.height * self.width)
                     if i not in forbidden]
        chosen    = torch.tensor(random.sample(available, self.num_mines))

        flat          = torch.zeros(self.height * self.width)
        flat[chosen]  = 1
        self.mines    = flat.reshape(self.height, self.width)
        self.numbers  = F.conv2d(
            self.mines.view(1, 1, self.height, self.width), kernel, padding=1
        ).squeeze()
        self._mines_placed = True

    # ── game logic ────────────────────────────────────────────────────────────

    def _neighbors(self, r, c):
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if (dr or dc) and 0 <= r+dr < self.height and 0 <= c+dc < self.width:
                    yield r+dr, c+dc

    def _reveal(self, r, c):
        stack = [(r, c)]
        while stack:
            r, c = stack.pop()
            if not self.hidden[r, c] or self.mines[r, c]:
                continue
            self.hidden[r, c] = False
            if self.numbers[r, c] == 0:
                stack += [(nr, nc) for nr, nc in self._neighbors(r, c)
                          if self.hidden[nr, nc]]

    def click(self, r: int, c: int) -> str:
        """
        Click a cell.

        Returns:
            'already'  — cell was already revealed (no action taken)
            'mine'     — hit a mine, game over
            'safe'     — revealed safely
            'won'      — this click completed the game
        """
        if not self.hidden[r, c]:
            return "already"

        if not self._mines_placed:
            self._place_mines(r, c)   # guaranteed safe for (r, c)

        self.move_count += 1

        if self.mines[r, c]:
            self.hidden[r, c] = False
            self.exploded = True
            return "mine"

        self._reveal(r, c)
        return "won" if self.is_won() else "safe"

    def is_won(self) -> bool:
        return bool((self.hidden == self.mines.bool()).all())

    def is_over(self) -> bool:
        return self.exploded or self.is_won()

    # ── tensor helpers ────────────────────────────────────────────────────────

    def get_clues_tensor(self) -> torch.Tensor:
        """(1, 1, H, W) clue values — -1 for hidden, 0-8 for revealed."""
        clues = torch.where(self.hidden, torch.tensor(-1.0), self.numbers)
        return clues.unsqueeze(0).unsqueeze(0)

    def get_hidden_tensor(self) -> torch.Tensor:
        """(1, 1, H, W) hidden mask — 1 = hidden, 0 = revealed."""
        return self.hidden.float().unsqueeze(0).unsqueeze(0)

    @property
    def num_hidden(self)       -> int: return int(self.hidden.sum())
    @property
    def num_hidden_mines(self) -> int: return int((self.hidden & self.mines.bool()).sum())


# ── Agent ─────────────────────────────────────────────────────────────────────

class EBMAgent:
    """Wraps the model + sampler into a Minesweeper-playing agent."""

    def __init__(self, model: MinesweeperEnergyModel,
                 sampler: EnergySampler, device: torch.device):
        self.model   = model
        self.sampler = sampler
        self.device  = device

    def get_mine_probs(self, game: MinesweeperGame) -> torch.Tensor:
        """Run the sampler and return (H, W) mine probabilities."""
        clues  = game.get_clues_tensor().to(self.device)
        hidden = game.get_hidden_tensor().to(self.device)
        board_state = clues_to_board_state(clues, hidden)
        mine_probs  = self.sampler.sample(board_state)    # (1, 1, H, W)
        return mine_probs.squeeze().cpu()                  # (H, W)

    def pick_cell(self, game: MinesweeperGame) -> tuple[int, int]:
        """
        Pick the hidden cell with the lowest predicted mine probability.
        Ties broken arbitrarily by argmin.
        """
        mine_probs  = self.get_mine_probs(game)
        hidden_mask = game.hidden.float()

        # Set revealed cells to 1e9 so argmin never picks them
        masked = mine_probs + (1.0 - hidden_mask) * 1e9
        flat   = masked.view(-1).argmin().item()
        return flat // game.width, flat % game.width


def load_agent_from_checkpoint(checkpoint_path: str, device: torch.device,
                               inference_steps: int = 15) -> EBMAgent:
    """
    Loads the model weights and the calibrated alpha_base from a saved checkpoint,
    then constructs the EnergySampler and EBMAgent correctly for evaluation.
    """
    # Load the dictionary from the .pt file
    # (weights_only=False might be needed depending on how you saved, but start with True for safety)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # 1. Set up the model
    model = MinesweeperEnergyModel().to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()

    # 2. Extract the calibrated alpha (fallback to 1.0 if not found)
    alpha_base = checkpoint.get("alpha_base", 1.0)
    print(f"Loaded checkpoint '{checkpoint_path}' (Epoch {checkpoint.get('epoch', '?')})")
    print(f"Using calibrated alpha_base: {alpha_base:.4f} for {inference_steps} inference steps.")

    # 3. Create the fixed Sampler
    sampler = EnergySampler(
        model=model,
        num_steps=inference_steps,
        alpha=alpha_base,
        noise_std=0.0  # Crucial: 0.0 noise so we sink to the bottom of the energy valley!
    )

    # 4. Wrap in the Agent
    agent = EBMAgent(model, sampler, device)

    return agent

# ── Evaluation statistics ─────────────────────────────────────────────────────

@dataclass
class GameResult:
    won:               bool
    exploded:          bool
    moves:             int
    total_hidden:      int     # hidden cells at start (after opening)
    cells_revealed:    int     # safe cells correctly revealed
    mines_hit:         int
    prob_at_hit:       Optional[float] = None


@dataclass
class EvalStats:
    results: list[GameResult] = field(default_factory=list)

    @property
    def n(self):            return len(self.results)
    @property
    def win_rate(self):     return sum(r.won for r in self.results) / self.n
    @property
    def avg_moves(self):    return sum(r.moves for r in self.results) / self.n
    @property
    def avg_coverage(self):
        return sum(r.cells_revealed / max(r.total_hidden, 1)
                   for r in self.results) / self.n

    def summary(self) -> str:
        return (f"Games: {self.n} | "
                f"Win rate: {self.win_rate*100:.1f}% | "
                f"Avg moves: {self.avg_moves:.1f} | "
                f"Avg coverage: {self.avg_coverage*100:.1f}%")


# ── Evaluation loop ───────────────────────────────────────────────────────────

def evaluate(agent: EBMAgent, num_games: int = 200,
             height: int = 16, width: int = 30, num_mines: int = 99,
             verbose: bool = True) -> EvalStats:
    stats = EvalStats()
    game  = MinesweeperGame(height, width, num_mines)

    for idx in range(num_games):
        game.reset()
        initial_hidden = game.num_hidden
        mines_hit      = 0
        prob_at_hit    = None

        while not game.is_over():
            r, c   = agent.pick_cell(game)
            probs  = agent.get_mine_probs(game)
            chosen_prob = probs[r, c].item()

            result = game.click(r, c)
            if result == "mine":
                mines_hit   = 1
                prob_at_hit = chosen_prob
                break

        cells_revealed = initial_hidden - game.num_hidden - mines_hit
        stats.results.append(GameResult(
            won            = game.is_won(),
            exploded       = game.exploded,
            moves          = game.move_count,
            total_hidden   = initial_hidden,
            cells_revealed = cells_revealed,
            mines_hit      = mines_hit,
            prob_at_hit    = prob_at_hit,
        ))

        if verbose and (idx + 1) % 10 == 0:
            print(f"  Game {idx+1:>4}/{num_games} — {stats.summary()}")

    return stats


# ── Visualisation ─────────────────────────────────────────────────────────────

from IPython.display import display, clear_output
import time

def visualise_game(agent: EBMAgent, height: int = 16, width: int = 30,
                   num_mines: int = 99, max_steps: int = 80,
                   save_path: Optional[str] = None) -> bool:
    """
    Play one game interactively, rendering the board and mine probability
    heatmap at every step using IPython display for notebook compatibility.
    """
    game        = MinesweeperGame(height, width, num_mines)
    clue_colors = ["blue", "green", "red", "darkblue",
                   "maroon", "cyan", "black", "gray"]

    def draw(ax_b, ax_h, title, chosen=None, result=None):
        for ax in (ax_b, ax_h):
            ax.clear()
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xticks(np.arange(width  + 1) - 0.5, minor=True)
            ax.set_yticks(np.arange(height + 1) - 0.5, minor=True)
            ax.grid(which="minor", color="black", linewidth=1)

        ax_b.set_title(title, fontsize=8)
        ax_b.set_xlim(-0.5, width - 0.5)
        ax_b.set_ylim(-0.5, height - 0.5)
        ax_b.invert_yaxis()

        # Draw board cells
        for r in range(height):
            for c in range(width):
                if game.hidden[r, c]:
                    face = "red" if (game.exploded and game.mines[r, c]) else "darkgray"
                    ax_b.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1, facecolor=face))
                    if game.mines[r, c]:
                        ax_b.text(c, r, "M", ha="center", va="center",
                                  color="white", fontsize=8, fontweight="bold")
                else:
                    ax_b.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1,
                                                  facecolor="lightgray"))
                    val = int(game.numbers[r, c])
                    if val > 0:
                        ax_b.text(c, r, str(val), ha="center", va="center",
                                  color=clue_colors[val-1], fontweight="bold",
                                  fontsize=10)

        # Highlight chosen cell
        if chosen is not None:
            cr, cc  = chosen
            colour  = "red" if result == "mine" else "lime"
            ax_b.add_patch(mpatches.Rectangle(
                (cc-.5, cr-.5), 1, 1,
                facecolor="none", edgecolor=colour, linewidth=2.5
            ))

        # Probability heatmap (hidden cells only)
        probs   = agent.get_mine_probs(game).numpy()
        display_probs = np.where(game.hidden.numpy(), probs, np.nan)
        ax_h.set_title("Mine probability", fontsize=8)
        ax_h.imshow(display_probs, vmin=0, vmax=1, cmap="RdYlGn_r",
                    origin="upper", aspect="equal")
        if chosen is not None:
            cr, cc = chosen
            ax_h.add_patch(mpatches.Rectangle(
                (cc-.5, cr-.5), 1, 1,
                facecolor="none", edgecolor="blue", linewidth=2.5
            ))

    # Temporarily turn off interactive plotting so it doesn't auto-spit empty figures
    plt.ioff()
    fig, (ax_b, ax_h) = plt.subplots(1, 2, figsize=(11, 5))
    plt.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn_r"),
                 ax=ax_h, fraction=0.046, pad=0.04, label="P(mine)")

    step = 0
    while not game.is_over() and step < max_steps:
        r, c   = agent.pick_cell(game)
        result = game.click(r, c)

        label = (f"Step {step+1} — clicked ({r},{c}) "
                 f"-> {'MINE' if result=='mine' else 'safe'}"
                 f"{'  WON!' if result=='won' else ''}")
        draw(ax_b, ax_h, label, chosen=(r, c), result=result)

        plt.tight_layout()
        if save_path:
            plt.savefig(f"{save_path}_step{step+1:02d}.png", dpi=100)

        # --- The Kaggle-safe animation magic ---
        clear_output(wait=True)
        display(fig)
        time.sleep(0.4)
        # ---------------------------------------

        step += 1

    # Final frame
    outcome = "WON" if game.is_won() else "LOST"
    draw(ax_b, ax_h, f"Game over — {outcome} in {game.move_count} moves")
    plt.tight_layout()
    if save_path:
        plt.savefig(f"{save_path}_final.png", dpi=100)

    clear_output(wait=True)
    display(fig)

    # Clean up and restore plotting defaults
    plt.close(fig)
    plt.ion()

    return game.is_won()


# ── System 2 benchmark ────────────────────────────────────────────────────────

def benchmark(model: MinesweeperEnergyModel, alpha_base: float,
              alpha_factor: float, device: torch.device,
              num_games: int = 300,
              steps_list: list[int] = [1, 5, 15, 30, 50]):
    """
    Test the same model at different inference step counts to produce the
    System 2 Thinking scaling curve from the paper (Figure 6a equivalent).

    A well-trained EBM should show monotonically increasing win rate as
    inference steps increase.
    """
    results = {}

    for n_steps in steps_list:
        print(f"\n── {n_steps} inference steps ──")
        sampler = EnergySampler(model, num_steps=n_steps, alpha=alpha_base)
        agent   = EBMAgent(model, sampler, device)
        stats   = evaluate(agent, num_games=num_games, verbose=True)
        results[n_steps] = stats
        print(f"   {stats.summary()}")

    # Plot
    win_rates = [results[s].win_rate * 100  for s in steps_list]
    coverages = [results[s].avg_coverage * 100 for s in steps_list]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(steps_list, win_rates, "o-", color="steelblue", linewidth=2, markersize=8)
    ax1.set_xlabel("Inference steps (NFEs)"); ax1.set_ylabel("Win rate (%)")
    ax1.set_title("System 2 Thinking — Win Rate vs Compute")
    ax1.grid(True, alpha=0.3); ax1.set_xticks(steps_list)

    ax2.plot(steps_list, coverages, "o-", color="seagreen", linewidth=2, markersize=8)
    ax2.set_xlabel("Inference steps (NFEs)"); ax2.set_ylabel("Board coverage (%)")
    ax2.set_title("System 2 Thinking — Coverage vs Compute")
    ax2.grid(True, alpha=0.3); ax2.set_xticks(steps_list)

    plt.suptitle(f"EBM Minesweeper — {num_games} games each", fontsize=12)
    plt.tight_layout()
    plt.savefig("system2_scaling.png", dpi=150)
    print("\nSaved → system2_scaling.png")
    plt.show()
    return results


# ── Diagnostic ────────────────────────────────────────────────────────────────

def diagnose(model: MinesweeperEnergyModel, sampler: EnergySampler,
             device: torch.device, n_boards: int = 5):
    """
    Three checks to determine whether the energy landscape formed correctly:

    1. E(true mines) < E(all mines)  — model correctly ranks the true answer
    2. Gradient magnitude doesn't collapse after the first inner step
    3. Sampler output has meaningful spread (std >> 0, mean != 0.5)
    """
    # from generate_dataset import generate_board

    model.eval()
    print("=" * 60)
    print("DIAGNOSTIC 1: Energy ranking")
    print("  E(true mines) should be LOWEST")
    print("=" * 60)

    correct = 0
    for i in range(n_boards):
        clues, mines, hidden = generate_board()
        clues  = clues.unsqueeze(0).to(device)
        mines  = mines.unsqueeze(0).to(device)
        hidden = hidden.unsqueeze(0).to(device)
        board  = clues_to_board_state(clues, hidden)

        with torch.no_grad():
            e_true = model(board, mines).item()
            e_all  = model(board, torch.ones_like(mines)).item()
            e_none = model(board, torch.zeros_like(mines)).item()
            e_rand = model(board, torch.rand_like(mines)).item()

        ok = e_true < e_all
        if ok: correct += 1
        print(f"\n  Board {i+1}:  E(true)={e_true:.2f}  E(all)={e_all:.2f}  "
              f"E(none)={e_none:.2f}  E(rand)={e_rand:.2f}  "
              f"{'OK' if ok else 'INVERTED'}")

    print(f"\n  Correct ranking: {correct}/{n_boards}")

    print("\n" + "=" * 60)
    print("DIAGNOSTIC 2: Gradient magnitude over inner steps")
    print("  Should stay > 0.001 — collapse means flat landscape")
    print("=" * 60)

    clues, mines, hidden = generate_board()
    clues  = clues.unsqueeze(0).to(device)
    hidden = hidden.unsqueeze(0).to(device)
    board  = clues_to_board_state(clues, hidden)

    with torch.enable_grad():
            cand = torch.randn(1, 1, 16, 30, device=device) * 0.1
            cand.requires_grad_(True)

            for s in range(6):
                probs  = torch.sigmoid(cand)
                energy = model(board, probs)
                grad   = torch.autograd.grad(energy.sum(), cand, create_graph=False)[0]

                print(f"  Step {s+1}: energy={energy.item():.4f}  "
                      f"grad_norm={grad.abs().mean().item():.6f}")

                with torch.no_grad():
                    # 2. Out-of-place update so PyTorch doesn't lose its mind
                    cand = cand - 1.0 * grad

                # 3. Re-enable tracking for the next loop iteration
                cand.requires_grad_(True)

    print("\n" + "=" * 60)
    print("DIAGNOSTIC 3: Sampler output distribution")
    print("  std should be >> 0.02  (0.02 = pure noise)")
    print("=" * 60)

    clues, mines, hidden = generate_board()
    clues  = clues.unsqueeze(0).to(device)
    hidden_t = hidden.unsqueeze(0).to(device)
    board  = clues_to_board_state(clues, hidden_t)

    probs_out    = sampler.sample(board).squeeze().cpu().numpy()
    hidden_np    = hidden.squeeze().numpy().astype(bool)
    hidden_probs = probs_out[hidden_np]

    print(f"  Hidden cell probs — min:{hidden_probs.min():.3f}  "
          f"max:{hidden_probs.max():.3f}  mean:{hidden_probs.mean():.3f}  "
          f"std:{hidden_probs.std():.3f}")

    # Also compare against random agent baseline
    print("\n" + "=" * 60)
    print("DIAGNOSTIC 4: vs random agent baseline")
    print("=" * 60)

    rand_wins, rand_moves, n = 0, 0, 200
    g = MinesweeperGame()
    for _ in range(n):
        g.reset()
        while not g.is_over():
            cells = g.hidden.nonzero().tolist()
            g.click(*random.choice(cells))
        rand_wins  += g.is_won()
        rand_moves += g.move_count

    print(f"  Random agent ({n} games): "
          f"win_rate={rand_wins/n*100:.1f}%  avg_moves={rand_moves/n:.1f}")
    model.train()

In [ ]:
create_and_save_dataset("minesweeper_train.pt", num_samples=100_000)

In [ ]:
train("minesweeper_train.pt", epochs=50)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt    = torch.load("checkpoints/ebm_epoch050.pt")
model   = MinesweeperEnergyModel().to(device)
model.load_state_dict(ckpt["model"])
sampler = EnergySampler(model, num_steps=15, alpha=ckpt["alpha_base"])

agent = load_agent_from_checkpoint("checkpoints/ebm_epoch050.pt", device, inference_steps=15)

evaluate(agent, num_games=200)
diagnose(model, sampler, device)
visualise_game(agent)
benchmark(model, ckpt["alpha_base"], ckpt["alpha_factor"], device)